# Deaths by Location and Triage

Loads a single-run `patient_log.xlsx`, identifies where each patient died
(last logged phase before the `DIED` event), and plots deaths by triage
category for each evacuation-chain location:

| Location | Meaning |
|---|---|
| **CCP — waiting for transport** | Patient at a CCP, waiting for a CCP→R1 ambulance |
| **R1 — waiting for care** | Patient arrived at Role 1, queued for DCR/PEC treatment |
| **R1 — waiting for transport** | Patient treated at Role 1, waiting for a R1→R2 ambulance |
| **R2 — waiting for care** | Patient arrived at Role 2, queued for DCS surgery |

In [ ]:
import sys
from pathlib import Path
_root = Path(".").resolve()
while not (_root / "paths.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))
import paths

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

run_folder = paths.SINGLE_RUN_OUTPUT / FOLDER_NAME

pl = pd.read_excel(run_folder / "patient_log.xlsx")
print(f"Loaded {len(pl)} log rows, {pl['patient_id'].nunique()} patients")

n_dead = pl.loc[pl["action"] == "DIED", "patient_id"].nunique()
n_safe = pl.loc[pl["action"] == "BECAME_SAFE", "patient_id"].nunique()
print(f"Dead: {n_dead}  |  Safe: {n_safe}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CLASSIFY DEATH LOCATION
# ═══════════════════════════════════════════════════════════════════════════
# For each dead patient we find the last WAITING_* action before DIED.
# We cannot simply use the row immediately before DIED because death may
# be discovered during pause_death_clock(), which logs CLOCK_PAUSED:*
# right before DIED.

PHASE_TO_LOCATION = {
    "WAITING_TRANSPORT:ccp":   "CCP — waiting for transport",
    "WAITING_CARE:dcr":        "R1 — waiting for care",
    "WAITING_CARE:pec":        "R1 — waiting for care",
    "WAITING_TRANSPORT:role1": "R1 — waiting for transport",
    "WAITING_CARE:dcs":        "R2 — waiting for care",
}

LOCATION_ORDER = [
    "CCP — waiting for transport",
    "R1 — waiting for care",
    "R1 — waiting for transport",
    "R2 — waiting for care",
]

dead_ids = pl.loc[pl["action"] == "DIED", "patient_id"].unique()
records = []
for pid in dead_ids:
    prows = pl[pl["patient_id"] == pid].sort_values("time")
    died_idx = prows.index[prows["action"] == "DIED"][0]
    pos = prows.index.get_loc(died_idx)
    # Scan backward to find the last WAITING_* action
    waiting_action = None
    for i in range(pos - 1, -1, -1):
        act = prows.iloc[i]["action"]
        if act in PHASE_TO_LOCATION:
            waiting_action = prows.iloc[i]
            break
    if waiting_action is None:
        continue
    triage = waiting_action["triage"]
    triage_short = triage.split("_")[0] if "_" in str(triage) else str(triage)
    location = PHASE_TO_LOCATION[waiting_action["action"]]
    records.append({"patient_id": pid, "triage": triage_short, "location": location})

deaths = pd.DataFrame(records)
print(f"Classified {len(deaths)} deaths:")
print(deaths.groupby("location")["patient_id"].count().reindex(LOCATION_ORDER).fillna(0).astype(int))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PLOT: one bar chart per location, T1/T2/T3 on x-axis
# ═══════════════════════════════════════════════════════════════════════════

TRIAGE_ORDER  = ["T1", "T2", "T3"]
TRIAGE_COLORS = {"T1": "#e74c3c", "T2": "#f39c12", "T3": "#3498db"}

fig, axes = plt.subplots(1, len(LOCATION_ORDER), figsize=(4 * len(LOCATION_ORDER), 5),
                         sharey=True)

for ax, loc in zip(axes, LOCATION_ORDER):
    sub = deaths[deaths["location"] == loc]
    counts = sub.groupby("triage")["patient_id"].count().reindex(TRIAGE_ORDER).fillna(0)
    bars = ax.bar(TRIAGE_ORDER, counts,
                  color=[TRIAGE_COLORS[t] for t in TRIAGE_ORDER],
                  edgecolor="black", width=0.6)
    for bar, val in zip(bars, counts):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    str(int(val)), ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_title(loc, fontsize=11)
    ax.set_xlabel("Triage")
    ax.set_ylim(bottom=0)

axes[0].set_ylabel("Number of deaths")
fig.suptitle(f"Deaths by location — {FOLDER_NAME}", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()